In [67]:
# Merge_FAG_JewishGen_Cemeteries.py
# 6/16/2025


from bs4 import BeautifulSoup 
import datetime
from datetime import datetime as dt
from datetime import date 
from datetime import timedelta
import time
import re
from pandas import read_csv
import pandas as pd
# from selenium import webdriver
from seleniumwire import webdriver
from selenium.webdriver.common.by import By
import json
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
import sys
import random
from IPython.display import display, HTML

def maximize_screen():
    from IPython.display import display, HTML
    display(HTML("<style>:root { --jp-notebook-max-width: 100% !important; }</style>"))


def display_dataframe_segments(dataframe_df):
    for i in range(0,len(dataframe_df),50):
        print(dataframe_df.iloc[i:i+50])
        
def clean_dataframe(dataframe_df):
    delete_list=['\(','\)','Cemetery','\*','\&',' County',' in','  ']
    for item in delete_list:
        dataframe_df= dataframe_df.astype(str).replace(item, ' ', regex=True).str.strip()
    return dataframe_df  
    
def clean_name(name):
    search_list=[' at ','Beth El','Bnai','Mt.']
    replace_list=['  ','Beth-El',"B'nai",'Mount']
    for search_item, replace_item in zip(search_list, replace_list):
        name = name.replace(search_item, replace_item).strip()    
    delete_list=['(',')','City','*','Cemetery','of','and','&',' County']
    for item in delete_list:
        name= name.replace(item, '').strip()    
    return name    

maximize_screen()
process_name='Merge FAG JewishGen Cemeteries'
DirMain="C:\\Users\\jrozm\\OneDrive\\Benjamin\\"

fileInputJG="Jewish_Gen_Cemeteries_X"
fileInputFAG="Benjamin_Cemeteries"
fileOutputJG="Jewish_Gen_Cemeteries_Mergered"


fileInJG=DirMain+"OutputData\\Cemeteries\\"+fileInputJG+".csv"
fileInFAG=DirMain+"OutputData\\Cemeteries\\"+fileInputFAG+".csv"
fileOutJG=DirMain+"OutputData\\Cemeteries\\"+fileOutputJG+".csv"
fileOutJGInt=DirMain+"OutputData\\Cemeteries\\"+fileOutputJG+"_Interval.csv"
cemeteries_jg_df=pd.read_csv(fileInJG)
cemeteries_fag_df=pd.read_csv(fileInFAG, encoding='ISO-8859-1')
clean_dataframe(cemeteries_fag_df['county'])

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', None)

name_count=0
total_cemetery_count=len(cemeteries_jg_df)
merged_jg_fag_cemeteries_df=pd.DataFrame([])
for index, row in cemeteries_jg_df.iterrows():
    name_count+=1
    cemetery_jg_name=row['cemetery_name']
    cemetery_jg_city=row['city']
    cemetery_jg_state=row['state']
    cemetery_jg_county=row['county']
    cemetery_jg_graves=row['graves']
    cemetery_jg_name=clean_name(cemetery_jg_name)
    jg_name_split=cemetery_jg_name.split(' ')
    jg_name_set=set(jg_name_split) 
    jg_name_set.discard('') 
    jg_name_set_size=len(jg_name_set)
     
    cemeteries_fag_df['county']=clean_dataframe(cemeteries_fag_df['county'])
    cemeteries_fag_name=cemeteries_fag_df['cemetery'][(cemeteries_fag_df['state']==cemetery_jg_state) & (cemeteries_fag_df['county']==cemetery_jg_county) ]
    cemeteries_fag_id=cemeteries_fag_df['id'][(cemeteries_fag_df['state']==cemetery_jg_state) & (cemeteries_fag_df['county']==cemetery_jg_county) ]
    cemeteries_fag_county=cemeteries_fag_df['county'][(cemeteries_fag_df['state']==cemetery_jg_state) & (cemeteries_fag_df['county']==cemetery_jg_county) ]
    cemeteries_fag_grave_count=cemeteries_fag_df['memorialCount'][(cemeteries_fag_df['state']==cemetery_jg_state) & (cemeteries_fag_df['county']==cemetery_jg_county) ]
    cemeteries_fag_name=clean_dataframe(cemeteries_fag_name)
    cemeteries_fag_name.reset_index(drop=True, inplace = True)
    fag_name_split_df=pd.DataFrame(cemeteries_fag_name.str.split(' '))    

    match_df=pd.DataFrame([])
    print(f"**** Searching: {cemetery_jg_name} • {cemetery_jg_city} • {cemetery_jg_state} • {cemetery_jg_county} ") 
    record_count=0
    
    for fag_name in fag_name_split_df['cemetery']:
        cemetery_id=cemeteries_fag_id.iloc[record_count]
        cemetery_county=cemeteries_fag_county.iloc[record_count]
        cemetery_fag_graves=cemeteries_fag_grave_count.iloc[record_count]
        fag_name_set=set(fag_name)
        missing_words=jg_name_set-fag_name_set
        matched_words=jg_name_set & fag_name_set
        missing_words_count=len(missing_words)
        matched_words_count=len(matched_words)
        match_percent=0
        if (matched_words_count+missing_words_count)>0:match_percent=int(100*matched_words_count/(matched_words_count+missing_words_count))
        cemetery_name_fields=['JG_Name','cemetery_jg_graves','FAG_Name','cemetery_fag_graves','matched_words','missing_words','cemetery_id','cemetery_jg_state','cemetery_jg_city','cemetery_county','match_percent']
        match_record_df=pd.DataFrame([cemetery_jg_name,cemetery_jg_graves,cemeteries_fag_name.iloc[record_count],cemetery_fag_graves,matched_words_count,missing_words_count,cemetery_id,cemetery_jg_state,cemetery_jg_city,cemetery_county,match_percent]).T
        match_record_df.columns=cemetery_name_fields
        match_df=pd.concat([match_df,match_record_df])
        record_count+=1
        
    if len(match_df)<=1:continue
    matched_best_df=match_df[(match_df['match_percent']==match_df['match_percent'].max())]
    best_matched_words_df=pd.DataFrame(matched_best_df['matched_words'])
    best_missing_words_df=pd.DataFrame(matched_best_df['missing_words'])	
    best_match_percent_df=pd.DataFrame(matched_best_df['match_percent'])
    merged_jg_fag_cemeteries_df=pd.concat([merged_jg_fag_cemeteries_df,matched_best_df])
    
    pd.set_option('display.width', 200)
    pd.set_option('display.max_columns', None)
    # print(matched_best_df)
    display(f"**** •  *{cemetery_jg_name}* •  {cemetery_jg_city} • {cemetery_jg_state} • {cemetery_jg_county}  •• Best Match: {matched_best_df['FAG_Name'].iloc[0]} - {best_match_percent_df['match_percent'].iloc[0]}% •• Process Count:  {name_count}/{total_cemetery_count}")
    print('')
    merged_jg_fag_cemeteries_df.to_csv(fileOutJGInt,mode="w", index=False,header=True)
merged_jg_fag_cemeteries_df.to_csv(fileOutJG,mode="w", index=False,header=True)
print('Uploaded To: ',fileOutJG)
    # collection_columns_set=set(list(collection_df))
    # missing_columns=field_list_set-collection_columns_set
    # collection_df[list(missing_columns)]=np.nan
# print(display_dataframe_segments(cemeteries_fag_name))
# input('')

    
# display_dataframe_segments(display_df)       
# relative_split0_df=pd.DataFrame(relative_df_split.str[0])
# first_name_3=relative_split0_df[relative_split0_df.iloc[:, 0].str.len() == 3]

# relative_df.to_csv(fileOut,mode="w", index=False,header=True)
# print("\n")
# print('Uploaded To: ',fileOut)
# display_dataframe_segments(display_df) 


**** Searching:  • Eagle Pass • Texas • Maverick 


'**** •  ** •  Eagle Pass • Texas • Maverick  •• Best Match: African American - 0% •• Process Count:  1/233'


**** Searching: B'nai Zion • El Paso • Texas • El Paso 


"**** •  *B'nai Zion* •  El Paso • Texas • El Paso  •• Best Match: B'nai Zion - 100% •• Process Count:  2/233"


**** Searching: Temple Mount Sinai • El Paso • Texas • El Paso 


...


**** Searching: Cheyenne   Complex   Mount Sinai • Cheyenne • Wyoming • Laramie 


'**** •  *Cheyenne   Complex   Mount Sinai* •  Cheyenne • Wyoming • Laramie  •• Best Match: Mount Sinai Jewish - 50% •• Process Count:  232/233'


**** Searching: Oregon Trail State Veterans • Evansville • Wyoming • Natrona 


'**** •  *Oregon Trail State Veterans* •  Evansville • Wyoming • Natrona  •• Best Match: Oregon Trail State Veterans - 100% •• Process Count:  233/233'


Uploaded To:  C:\Users\jrozm\OneDrive\Benjamin\OutputData\Cemeteries\Jewish_Gen_Cemeteries_Mergered.csv


In [65]:
        cemetery_id=cemeteries_fag_id.iloc[record_count]
        cemetery_county=cemeteries_fag_county.iloc[record_count]
        cemetery_fag_graves=cemeteries_fag_grave_count.iloc[record_count]
        fag_name_set=set(fag_name)
        missing_words=jg_name_set-fag_name_set
        matched_words=jg_name_set & fag_name_set
        missing_words_count=len(missing_words)
        matched_words_count=len(matched_words)
        match_percent=0
        if (matched_words_count+missing_words_count)>0:match_percent=int(100*matched_words_count/(matched_words_count+missing_words_count))
        cemetery_name_fields=['JG_Name','cemetery_jg_graves','FAG_Name','cemetery_fag_graves','matched_words','missing_words','cemetery_id','cemetery_jg_state','cemetery_jg_city','cemetery_county','match_percent']
        match_record_df=pd.DataFrame([cemetery_jg_name,cemetery_jg_graves,cemeteries_fag_name.iloc[record_count],cemetery_fag_graves,matched_words_count,missing_words_count,cemetery_id,cemetery_jg_state,cemetery_jg_city,cemetery_county,match_percent]).T
        match_record_df.columns=cemetery_name_fields
        match_df=pd.concat([match_df,match_record_df])
        record_count+=1
        
    if len(match_df)<=1:continue
    matched_best_df=match_df[(match_df['match_percent']==match_df['match_percent'].max())]
    best_matched_words_df=pd.DataFrame(matched_best_df['matched_words'])
    best_missing_words_df=pd.DataFrame(matched_best_df['missing_words'])	
    best_match_percent_df=pd.DataFrame(matched_best_df['match_percent'])
    merged_jg_fag_cemeteries_df=pd.concat([merged_jg_fag_cemeteries_df,matched_best_df])
    
    pd.set_option('display.width', 200)
    pd.set_option('display.max_columns', None)
    # print(matched_best_df)
    display(f"**** •  *{cemetery_jg_name}* •  {cemetery_jg_city} • {cemetery_jg_state} • {cemetery_jg_county}  •• Best Match: {matched_best_df['FAG_Name'].iloc[0]} - {best_match_percent_df['match_percent'].iloc[0]}% •• Process Count:  {name_count}/{total_cemetery_count}")
    print('')
    merged_jg_fag_cemeteries_df.to_csv(fileOutJGInt,mode="w", index=False,header=True)


IndentationError: unexpected indent (161095815.py, line 17)

In [18]:
def clean_dataframe(dataframe_df):
    delete_list=['\(','\)','Cemetery','\*','\&',' County',' in','  ']
    for item in delete_list:
        dataframe_df= dataframe_df.astype(str).replace(item, ' ', regex=True).str.strip()
    return dataframe_df  
    
cemeteries_fag_name=clean_dataframe(cemeteries_fag_name)
cemeteries_fag_name_df=pd.DataFrame(cemeteries_fag_name.str.split(' '))
cemeteries_fag_name_df

,cemetery
0,"[10th, Street]"
1,"[Alexander, Family]"
2,"[Alexandria, United, Methodist, Church]"
3,[Allen]
4,"[Anniston, Memorial, Gardens]"
...,...
258,"[Woodrow, W., Smith, Gravesite]"
259,"[Woolf, Family]"
260,"[Youngs, Chapel, Methodist, Church]"
261,"[edge, mount, north, Anniston, Ala, 36201]"
